# R3.09 - SAÉ3.01 : Mot de passe

**Étudiant :** Mamadou-Bailo BARRY - BUT Informatique 2 (Site d'Arles)
**Enseignant :** M. Bruno Colombel
**Environnement :** Python 3 (Jupyter)

## 1 Avertissements

On importe les bibliothèques utilisées dans le TP. Comme demandé dans le bonus, la ligne `from Crypto.Cipher import AES` est mise au début du script.

In [ ]:
import hashlib
import sqlite3
import bcrypt
import time
from random import getrandbits, choice
from Crypto.Cipher import AES

### 1.1 Encodage

Exemple du sujet : `encode` transforme une chaîne (`str`) en chaîne binaire (`bytes`, avec un `b` devant), `decode` fait l'inverse.

In [ ]:
txt="Il était une fois..."
print(txt)
print(txt.encode("utf-8"))
print(txt.encode("latin1"))
print("\n")

txt="Ben ça alors !".encode("utf-8")
print(txt)
print(type(txt))
print(txt.decode())#La valeur par défaut est celle de jupyter = celle du navigateur
                   #en général UTF8

La dernière instruction du sujet, `txt.decode("ascii")`, provoque une erreur : le `ç` est codé en UTF-8 par l'octet `\xc3` (195), qui n'existe pas en ASCII. On affiche l'erreur avec `try` / `except` pour que la suite du notebook continue de s'exécuter :

In [ ]:
try:
    print(txt.decode("ascii"))
except UnicodeDecodeError as erreur:
    print("Erreur :", erreur)

### 1.2 Fonctions de hachage

Les fonctions de `hashlib` prennent des `bytes` ; `.hexdigest()` donne l'empreinte en hexadécimal (64 caractères pour SHA-256).

In [ ]:
s = 'bonjour '.encode('utf-8')
s2 = 'bonjour !'.encode('utf-8')

In [ ]:
s.hex(), s2.hex()

In [ ]:
hashlib.sha256(s).hexdigest(), hashlib.sha256(s2).hexdigest()

### Exercice 1

Il y a $26^5 = 11\,881\,376$ mots de 5 lettres minuscules. On les génère tous avec 5 boucles imbriquées et on range les hachés dans une table de hachage (dictionnaire) `haché → mot` : c'est une *Rainbow Table* (cours, niveau de sécurité $-2$).

⚠️ La construction prend une vingtaine de secondes et environ 2,4 Go de mémoire.

In [ ]:
alphabet = "abcdefghijklmnopqrstuvwxyz"

table = {}
for a in alphabet:
    for b in alphabet:
        for c in alphabet:
            for d in alphabet:
                for e in alphabet:
                    mot = a + b + c + d + e
                    table[hashlib.sha256(mot.encode('utf-8')).hexdigest()] = mot

len(table)

Une fois la table construite, la recherche du mot de passe est immédiate :

In [ ]:
table["9e69e7e29351ad837503c44a5971edebc9b7e6d8601c89c284b1b59bf37afa80"]

Le mot de passe est **`qwert`**. On libère la mémoire occupée par la table avant de continuer :

In [ ]:
del table

## 2 Stockage de mots de passe

In [ ]:
# ouvrir la connexion ou créer la base
connection = sqlite3.connect("donnees.db")

# Créer un curseur
curseur = connection.cursor()

# Créer la table
curseur.execute("CREATE TABLE IF NOT EXISTS utilisateurs (name TEXT, password TEXT)")

### Exercice 2 : Niveau de sécurité $-\infty$

Fonction `AjoutUtilisateur()` donnée dans le sujet : le mot de passe est stocké **en clair** (« Interdit !!! » d'après le cours).

In [ ]:
def AjoutUtilisateur() :
    nom = input("login : ")
    curseur.execute("SELECT name FROM utilisateurs WHERE name == '" + nom + "'")
    res = curseur.fetchall()
    if (nom,) in res :
        print("Utilisateur déjà inscrit")
    else :
        mot_de_passe = input("Entrer votre mot de passe : ")
        mot_de_passe_bis = input("Entrer à nouveau votre mot de passe : ")
        if mot_de_passe == mot_de_passe_bis :
            curseur.execute("INSERT INTO utilisateurs (name, password) VALUES (?, ?)", (nom, mot_de_passe))
            connection.commit()
        else :
            print("Erreur de saisie")

On l'exécute au moins une fois (par exemple login `alice`, mot de passe `soleil`) :

In [ ]:
AjoutUtilisateur()

1. On affiche toutes les données de la base : le mot de passe est lisible directement.

In [ ]:
curseur.execute("SELECT * FROM utilisateurs")
curseur.fetchall()

2. `Verification()` : même schéma que `AjoutUtilisateur()`. `fetchall()` renvoie une liste de tuples : `[]` si le login n'existe pas, sinon `res[0][0]` est le mot de passe stocké.

*Remarque :* on utilise des `?` dans les requêtes, comme dans l'`INSERT` du sujet, plutôt que de coller le login dans la requête.

In [ ]:
def Verification() :
    nom = input("login : ")
    mot_de_passe = input("Entrer votre mot de passe : ")
    curseur.execute("SELECT password FROM utilisateurs WHERE name == ?", (nom,))
    res = curseur.fetchall()
    if res == [] :
        print("Utilisateur inconnu")
    elif res[0][0] == mot_de_passe :
        print("Authentification réussie")
    else :
        print("Mot de passe incorrect")

In [ ]:
Verification()

### Exercice 3 : Niveau de sécurité $-2$

On stocke `SHA256(mot_de_passe)`. **Attention au format :** `sha256` prend des `bytes` (d'où `.encode('utf-8')`) et `.hexdigest()` renvoie une `str`, qu'on peut stocker et comparer avec `==`.

In [ ]:
curseur.execute("CREATE TABLE IF NOT EXISTS utilisateurs_sha256 (name TEXT, password TEXT)")

def AjoutUtilisateur_sha256() :
    nom = input("login : ")
    curseur.execute("SELECT name FROM utilisateurs_sha256 WHERE name == ?", (nom,))
    res = curseur.fetchall()
    if (nom,) in res :
        print("Utilisateur déjà inscrit")
    else :
        mot_de_passe = input("Entrer votre mot de passe : ")
        mot_de_passe_bis = input("Entrer à nouveau votre mot de passe : ")
        if mot_de_passe == mot_de_passe_bis :
            hache = hashlib.sha256(mot_de_passe.encode('utf-8')).hexdigest()
            curseur.execute("INSERT INTO utilisateurs_sha256 (name, password) VALUES (?, ?)", (nom, hache))
            connection.commit()
        else :
            print("Erreur de saisie")

def Verification_sha256() :
    nom = input("login : ")
    mot_de_passe = input("Entrer votre mot de passe : ")
    curseur.execute("SELECT password FROM utilisateurs_sha256 WHERE name == ?", (nom,))
    res = curseur.fetchall()
    if res == [] :
        print("Utilisateur inconnu")
    elif res[0][0] == hashlib.sha256(mot_de_passe.encode('utf-8')).hexdigest() :
        print("Authentification réussie")
    else :
        print("Mot de passe incorrect")

In [ ]:
AjoutUtilisateur_sha256()

In [ ]:
curseur.execute("SELECT * FROM utilisateurs_sha256")
curseur.fetchall()

In [ ]:
Verification_sha256()

Faiblesse : deux mots de passe identiques ont le même haché, et une *Rainbow Table* des mots de passe courants les retrouve tout de suite (comme à l'exercice 1).

### Exercice 4 : Niveau de sécurité $-1$

On ajoute un **sel global**, écrit en dur dans le code : `SHA256(mot_de_passe + SEL)`. Les *Rainbow Tables* existantes ne marchent plus, mais un attaquant qui retrouve le sel (un seul pour tout le monde) peut en construire une nouvelle.

In [ ]:
SEL = "R3.09-SAE3.01"
curseur.execute("CREATE TABLE IF NOT EXISTS utilisateurs_saltedsha256 (name TEXT, password TEXT)")

def AjoutUtilisateur_saltedsha256() :
    nom = input("login : ")
    curseur.execute("SELECT name FROM utilisateurs_saltedsha256 WHERE name == ?", (nom,))
    res = curseur.fetchall()
    if (nom,) in res :
        print("Utilisateur déjà inscrit")
    else :
        mot_de_passe = input("Entrer votre mot de passe : ")
        mot_de_passe_bis = input("Entrer à nouveau votre mot de passe : ")
        if mot_de_passe == mot_de_passe_bis :
            hache = hashlib.sha256((mot_de_passe + SEL).encode('utf-8')).hexdigest()
            curseur.execute("INSERT INTO utilisateurs_saltedsha256 (name, password) VALUES (?, ?)", (nom, hache))
            connection.commit()
        else :
            print("Erreur de saisie")

def Verification_saltedsha256() :
    nom = input("login : ")
    mot_de_passe = input("Entrer votre mot de passe : ")
    curseur.execute("SELECT password FROM utilisateurs_saltedsha256 WHERE name == ?", (nom,))
    res = curseur.fetchall()
    if res == [] :
        print("Utilisateur inconnu")
    elif res[0][0] == hashlib.sha256((mot_de_passe + SEL).encode('utf-8')).hexdigest() :
        print("Authentification réussie")
    else :
        print("Mot de passe incorrect")

In [ ]:
AjoutUtilisateur_saltedsha256()

In [ ]:
curseur.execute("SELECT * FROM utilisateurs_saltedsha256")
curseur.fetchall()

In [ ]:
Verification_saltedsha256()

### Exercice 5 : Niveau de sécurité $0$

Un **sel aléatoire par utilisateur**, stocké dans une autre base (`sels.db`).

Sel de 16 octets = 128 bits : `getrandbits(128)`, puis `hex()` (on enlève le `0x` avec `[2:]`). `hex()` n'écrit pas les 0 de gauche : on complète à gauche avec des `0` jusqu'à 32 caractères.

*Remarque :* le sujet donne les mêmes noms de fonctions aux exercices 4 et 5 ; pour garder les deux versions, celles de l'exercice 5 s'appellent `..._randsaltedsha256()`.

In [ ]:
def nouveau_sel() :
    sel = hex(getrandbits(128))[2:]
    while len(sel) < 32 :
        sel = "0" + sel
    return sel

In [ ]:
nouveau_sel()

In [ ]:
connection_sel = sqlite3.connect("sels.db")
curseur_sel = connection_sel.cursor()
curseur_sel.execute("CREATE TABLE IF NOT EXISTS sels (name TEXT, salt TEXT)")
curseur.execute("CREATE TABLE IF NOT EXISTS utilisateurs_randsaltedsha256 (name TEXT, password TEXT)")

def AjoutUtilisateur_randsaltedsha256() :
    nom = input("login : ")
    curseur.execute("SELECT name FROM utilisateurs_randsaltedsha256 WHERE name == ?", (nom,))
    res = curseur.fetchall()
    if (nom,) in res :
        print("Utilisateur déjà inscrit")
    else :
        mot_de_passe = input("Entrer votre mot de passe : ")
        mot_de_passe_bis = input("Entrer à nouveau votre mot de passe : ")
        if mot_de_passe == mot_de_passe_bis :
            sel = nouveau_sel()
            hache = hashlib.sha256((mot_de_passe + sel).encode('utf-8')).hexdigest()
            curseur.execute("INSERT INTO utilisateurs_randsaltedsha256 (name, password) VALUES (?, ?)", (nom, hache))
            connection.commit()
            curseur_sel.execute("INSERT INTO sels (name, salt) VALUES (?, ?)", (nom, sel))
            connection_sel.commit()
        else :
            print("Erreur de saisie")

def Verification_randsaltedsha256() :
    nom = input("login : ")
    mot_de_passe = input("Entrer votre mot de passe : ")
    curseur.execute("SELECT password FROM utilisateurs_randsaltedsha256 WHERE name == ?", (nom,))
    res = curseur.fetchall()
    curseur_sel.execute("SELECT salt FROM sels WHERE name == ?", (nom,))
    res_sel = curseur_sel.fetchall()
    if res == [] or res_sel == [] :
        print("Utilisateur inconnu")
    elif res[0][0] == hashlib.sha256((mot_de_passe + res_sel[0][0]).encode('utf-8')).hexdigest() :
        print("Authentification réussie")
    else :
        print("Mot de passe incorrect")

In [ ]:
AjoutUtilisateur_randsaltedsha256()

In [ ]:
curseur.execute("SELECT * FROM utilisateurs_randsaltedsha256")
print(curseur.fetchall())
curseur_sel.execute("SELECT * FROM sels")
print(curseur_sel.fetchall())

In [ ]:
Verification_randsaltedsha256()

### Exercice 6 : Niveau de sécurité $1$

`bcrypt` (cours : basé sur Blowfish, **lent** à calculer, sel aléatoire).

- `bcrypt.gensalt()` génère le sel, `bcrypt.hashpw(mdp, sel)` hache (`bytes` → `bytes`), `bcrypt.checkpw(mdp, haché)` vérifie.
- Le sel est stocké **dans** le haché (`$2b$12$` + 22 caractères de sel + haché) : pas besoin d'une autre base.
- On ne compare pas avec `==` (le sel est aléatoire, deux hachages du même mot de passe sont différents) : c'est `checkpw` qui vérifie.

In [ ]:
curseur.execute("CREATE TABLE IF NOT EXISTS utilisateurs_bcrypt (name TEXT, password TEXT)")

def AjoutUtilisateur_bcrypt() :
    nom = input("login : ")
    curseur.execute("SELECT name FROM utilisateurs_bcrypt WHERE name == ?", (nom,))
    res = curseur.fetchall()
    if (nom,) in res :
        print("Utilisateur déjà inscrit")
    else :
        mot_de_passe = input("Entrer votre mot de passe : ")
        mot_de_passe_bis = input("Entrer à nouveau votre mot de passe : ")
        if mot_de_passe == mot_de_passe_bis :
            hache = bcrypt.hashpw(mot_de_passe.encode('utf-8'), bcrypt.gensalt())
            curseur.execute("INSERT INTO utilisateurs_bcrypt (name, password) VALUES (?, ?)", (nom, hache.decode('utf-8')))
            connection.commit()
        else :
            print("Erreur de saisie")

def Verification_bcrypt() :
    nom = input("login : ")
    mot_de_passe = input("Entrer votre mot de passe : ")
    curseur.execute("SELECT password FROM utilisateurs_bcrypt WHERE name == ?", (nom,))
    res = curseur.fetchall()
    if res == [] :
        print("Utilisateur inconnu")
    elif bcrypt.checkpw(mot_de_passe.encode('utf-8'), res[0][0].encode('utf-8')) :
        print("Authentification réussie")
    else :
        print("Mot de passe incorrect")

In [ ]:
AjoutUtilisateur_bcrypt()

In [ ]:
curseur.execute("SELECT * FROM utilisateurs_bcrypt")
curseur.fetchall()

In [ ]:
Verification_bcrypt()

## 3 Comparaison

### Exercice 7

Pour ne pas mélanger avec les comptes créés à la main, on travaille dans une nouvelle base `comparaison.db`.

In [ ]:
connection = sqlite3.connect("comparaison.db")
curseur = connection.cursor()

1. `generermdp(N, n, alph)` : `N` utilisateurs, mots de passe de `n` caractères tirés avec `choice(alph)`, stockés en clair dans la table `utilisateurs`.

In [ ]:
def generermdp(N, n, alph) :
    curseur.execute("DROP TABLE IF EXISTS utilisateurs")
    curseur.execute("CREATE TABLE utilisateurs (name TEXT, password TEXT)")
    for i in range(N) :
        nom = "user" + str(i)
        mot_de_passe = ""
        for j in range(n) :
            mot_de_passe = mot_de_passe + choice(alph)
        curseur.execute("INSERT INTO utilisateurs (name, password) VALUES (?, ?)", (nom, mot_de_passe))
    connection.commit()

In [ ]:
generermdp(50, 8, "abcdefghijklmnopqrstuvwxyz0123456789")
curseur.execute("SELECT * FROM utilisateurs LIMIT 5")
curseur.fetchall()

2. Une fonction de stockage par type : on recopie toute la table `utilisateurs` dans une nouvelle table, avec le mot de passe transformé comme aux exercices 2 à 6.

In [ ]:
def stockage_clair() :
    curseur.execute("DROP TABLE IF EXISTS stock_clair")
    curseur.execute("CREATE TABLE stock_clair (name TEXT, password TEXT)")
    curseur.execute("SELECT name, password FROM utilisateurs")
    for (nom, mot_de_passe) in curseur.fetchall() :
        curseur.execute("INSERT INTO stock_clair (name, password) VALUES (?, ?)", (nom, mot_de_passe))
    connection.commit()

def stockage_sha256() :
    curseur.execute("DROP TABLE IF EXISTS stock_sha256")
    curseur.execute("CREATE TABLE stock_sha256 (name TEXT, password TEXT)")
    curseur.execute("SELECT name, password FROM utilisateurs")
    for (nom, mot_de_passe) in curseur.fetchall() :
        hache = hashlib.sha256(mot_de_passe.encode('utf-8')).hexdigest()
        curseur.execute("INSERT INTO stock_sha256 (name, password) VALUES (?, ?)", (nom, hache))
    connection.commit()

def stockage_saltedsha256() :
    curseur.execute("DROP TABLE IF EXISTS stock_saltedsha256")
    curseur.execute("CREATE TABLE stock_saltedsha256 (name TEXT, password TEXT)")
    curseur.execute("SELECT name, password FROM utilisateurs")
    for (nom, mot_de_passe) in curseur.fetchall() :
        hache = hashlib.sha256((mot_de_passe + SEL).encode('utf-8')).hexdigest()
        curseur.execute("INSERT INTO stock_saltedsha256 (name, password) VALUES (?, ?)", (nom, hache))
    connection.commit()

def stockage_randsaltedsha256() :
    curseur.execute("DROP TABLE IF EXISTS stock_randsaltedsha256")
    curseur.execute("CREATE TABLE stock_randsaltedsha256 (name TEXT, password TEXT)")
    curseur_sel.execute("DROP TABLE IF EXISTS stock_sels")
    curseur_sel.execute("CREATE TABLE stock_sels (name TEXT, salt TEXT)")
    curseur.execute("SELECT name, password FROM utilisateurs")
    for (nom, mot_de_passe) in curseur.fetchall() :
        sel = nouveau_sel()
        hache = hashlib.sha256((mot_de_passe + sel).encode('utf-8')).hexdigest()
        curseur.execute("INSERT INTO stock_randsaltedsha256 (name, password) VALUES (?, ?)", (nom, hache))
        curseur_sel.execute("INSERT INTO stock_sels (name, salt) VALUES (?, ?)", (nom, sel))
    connection.commit()
    connection_sel.commit()

def stockage_bcrypt() :
    curseur.execute("DROP TABLE IF EXISTS stock_bcrypt")
    curseur.execute("CREATE TABLE stock_bcrypt (name TEXT, password TEXT)")
    curseur.execute("SELECT name, password FROM utilisateurs")
    for (nom, mot_de_passe) in curseur.fetchall() :
        hache = bcrypt.hashpw(mot_de_passe.encode('utf-8'), bcrypt.gensalt())
        curseur.execute("INSERT INTO stock_bcrypt (name, password) VALUES (?, ?)", (nom, hache.decode('utf-8')))
    connection.commit()

In [ ]:
stockage_clair()
stockage_sha256()
stockage_saltedsha256()
stockage_randsaltedsha256()
stockage_bcrypt()

3. Authentification sur la seule donnée du login : le mot de passe en clair est lu dans `utilisateurs`, puis vérifié comme aux exercices précédents. Les fonctions renvoient `True` ou `False`.

In [ ]:
def mdp_clair(nom) :
    curseur.execute("SELECT password FROM utilisateurs WHERE name == ?", (nom,))
    return curseur.fetchall()[0][0]

def auth_clair(nom) :
    mot_de_passe = mdp_clair(nom)
    curseur.execute("SELECT password FROM stock_clair WHERE name == ?", (nom,))
    return curseur.fetchall()[0][0] == mot_de_passe

def auth_sha256(nom) :
    mot_de_passe = mdp_clair(nom)
    curseur.execute("SELECT password FROM stock_sha256 WHERE name == ?", (nom,))
    return curseur.fetchall()[0][0] == hashlib.sha256(mot_de_passe.encode('utf-8')).hexdigest()

def auth_saltedsha256(nom) :
    mot_de_passe = mdp_clair(nom)
    curseur.execute("SELECT password FROM stock_saltedsha256 WHERE name == ?", (nom,))
    return curseur.fetchall()[0][0] == hashlib.sha256((mot_de_passe + SEL).encode('utf-8')).hexdigest()

def auth_randsaltedsha256(nom) :
    mot_de_passe = mdp_clair(nom)
    curseur_sel.execute("SELECT salt FROM stock_sels WHERE name == ?", (nom,))
    sel = curseur_sel.fetchall()[0][0]
    curseur.execute("SELECT password FROM stock_randsaltedsha256 WHERE name == ?", (nom,))
    return curseur.fetchall()[0][0] == hashlib.sha256((mot_de_passe + sel).encode('utf-8')).hexdigest()

def auth_bcrypt(nom) :
    mot_de_passe = mdp_clair(nom)
    curseur.execute("SELECT password FROM stock_bcrypt WHERE name == ?", (nom,))
    return bcrypt.checkpw(mot_de_passe.encode('utf-8'), curseur.fetchall()[0][0].encode('utf-8'))

In [ ]:
auth_clair("user0"), auth_sha256("user0"), auth_saltedsha256("user0"), auth_randsaltedsha256("user0"), auth_bcrypt("user0")

4. On mesure avec `time.time()` le temps d'authentification de tous les utilisateurs, pour chaque type de stockage.

In [ ]:
def temps_authentification(auth) :
    curseur.execute("SELECT name FROM utilisateurs")
    noms = curseur.fetchall()
    debut = time.time()
    for (nom,) in noms :
        if not auth(nom) :
            print("Échec pour", nom)
    return time.time() - debut

In [ ]:
print("clair                   :", temps_authentification(auth_clair))
print("SHA-256                 :", temps_authentification(auth_sha256))
print("SHA-256 + sel global    :", temps_authentification(auth_saltedsha256))
print("SHA-256 + sel aléatoire :", temps_authentification(auth_randsaltedsha256))
print("bcrypt                  :", temps_authentification(auth_bcrypt))

**Interprétation.**
- Les 4 premiers stockages sont quasi instantanés : SHA-256 est conçu pour être rapide.
- bcrypt prend de l'ordre de **10 secondes pour 50 utilisateurs** (environ 0,2 s par vérification), comme l'indique le sujet.
- 0,2 s ne gêne pas un utilisateur qui se connecte, mais pour une attaque par force brute c'est énorme : les $11\,881\,376$ mots de l'exercice 1 prendraient environ $11\,881\,376 \times 0{,}2 \approx 2{,}4$ millions de secondes (environ 27 jours), et cela pour **un seul** utilisateur puisque chaque sel est différent.

### Bonus : Niveau de sécurité 2

Cours : `AES(bcrypt($perUserSalt, $pwd), $secretKey)`. On chiffre le haché bcrypt avec AES en mode CTR et la clé `YELLOW SUBMARINE`. Le `nonce` tiré par le mode CTR est stocké dans un champ supplémentaire, et le chiffré comme le nonce sont stockés en hexadécimal (`.hex()` / `bytes.fromhex()`).

*Remarque :* `YELLOW SUBMARINE` fait 16 octets = 128 bits, donc c'est en fait de l'AES-128 (l'AES-256 demanderait une clé de 32 octets). On garde la clé imposée par le sujet.

On revient d'abord sur la base `donnees.db` (l'exercice 7 travaillait dans `comparaison.db`) :

In [ ]:
connection = sqlite3.connect("donnees.db")
curseur = connection.cursor()

In [ ]:
CLE = b"YELLOW SUBMARINE"
curseur.execute("CREATE TABLE IF NOT EXISTS utilisateurs_aes (name TEXT, password TEXT, nonce TEXT)")

def AjoutUtilisateur_aes() :
    nom = input("login : ")
    curseur.execute("SELECT name FROM utilisateurs_aes WHERE name == ?", (nom,))
    res = curseur.fetchall()
    if (nom,) in res :
        print("Utilisateur déjà inscrit")
    else :
        mot_de_passe = input("Entrer votre mot de passe : ")
        mot_de_passe_bis = input("Entrer à nouveau votre mot de passe : ")
        if mot_de_passe == mot_de_passe_bis :
            hache = bcrypt.hashpw(mot_de_passe.encode('utf-8'), bcrypt.gensalt())
            cipher = AES.new(CLE, AES.MODE_CTR)
            chiffre = cipher.encrypt(hache)
            curseur.execute("INSERT INTO utilisateurs_aes (name, password, nonce) VALUES (?, ?, ?)",
                            (nom, chiffre.hex(), cipher.nonce.hex()))
            connection.commit()
        else :
            print("Erreur de saisie")

def Verification_aes() :
    nom = input("login : ")
    mot_de_passe = input("Entrer votre mot de passe : ")
    curseur.execute("SELECT password, nonce FROM utilisateurs_aes WHERE name == ?", (nom,))
    res = curseur.fetchall()
    if res == [] :
        print("Utilisateur inconnu")
    else :
        chiffre = bytes.fromhex(res[0][0])
        nonce = bytes.fromhex(res[0][1])
        cipher = AES.new(CLE, AES.MODE_CTR, nonce=nonce)
        hache = cipher.decrypt(chiffre)
        if bcrypt.checkpw(mot_de_passe.encode('utf-8'), hache) :
            print("Authentification réussie")
        else :
            print("Mot de passe incorrect")

In [ ]:
AjoutUtilisateur_aes()

In [ ]:
curseur.execute("SELECT * FROM utilisateurs_aes")
curseur.fetchall()

In [ ]:
Verification_aes()